# Conformal predictive systems for time series

This notebook covers discrete demand and continuous measurements with Nixtla-compatible panel forecasters. Both CPS variants calibrate series- and horizon-specific residual distributions and return a self-contained, panel-aligned forecast.

In [1]:
import os
import sys
sys.path.append(os.path.abspath("../.."))

import numpy as np
import pandas as pd
from IPython.display import display
from mlforecast import MLForecast
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

from tinyconformal.series import (
    ContinuousTimeSeriesConformalPredictiveSystem,
    DiscreteTimeSeriesConformalPredictiveSystem,
)
from tinyconformal.utils import FirstStageEvaluator, NewsvendorSolver

pd.set_option("display.max_columns", 20)

## 1. Discrete demand

The target is a non-negative integer count. The last seven days are held out so evaluation uses observations that were not used during fitting.

In [2]:
def make_count_panel(n_periods=140, seed=42):
    rng = np.random.default_rng(seed)
    dates = pd.date_range("2025-01-01", periods=n_periods, freq="D")
    frames = []
    for offset, unique_id in enumerate(["store_A", "store_B"]):
        mean = 4.0 + offset + np.linspace(0, 1.5, n_periods) + 1.5 * (dates.dayofweek >= 5)
        frames.append(pd.DataFrame({
            "unique_id": unique_id, "ds": dates, "y": rng.poisson(mean)
        }))
    return pd.concat(frames, ignore_index=True)

horizon = 7
count_data = make_count_panel()
count_train = count_data.groupby("unique_id", group_keys=False).head(-horizon)
count_test = count_data.groupby("unique_id", group_keys=False).tail(horizon)
count_data.head()

,unique_id,ds,y
0,store_A,2025-01-01,6
1,store_A,2025-01-02,3
2,store_A,2025-01-03,6
3,store_A,2025-01-04,9
4,store_A,2025-01-05,7


In [3]:
count_learner = MLForecast(
    models={"LinearRegression": LinearRegression()},
    freq="D", lags=[1, 7, 14], date_features=["dayofweek"],
)
count_cps = DiscreteTimeSeriesConformalPredictiveSystem(
    learner=count_learner,
    dispersion_learner=RandomForestRegressor(
        n_estimators=100, min_samples_leaf=3, random_state=42, n_jobs=-1
    ),
    minimum=0,
).fit(count_train, horizon=horizon, n_windows=5, static_features=[], n_jobs=1)
count_forecast = count_cps.predict_distribution(h=horizon)
count_forecast.to_frame().head()

,unique_id,ds,LinearRegression
0,store_A,2025-05-14,4.420151
1,store_A,2025-05-15,4.512866
2,store_A,2025-05-16,4.912694
3,store_A,2025-05-17,5.357449
4,store_A,2025-05-18,5.902066


### Discrete PPF, CDF, SF, and PMF

Each method returns a DataFrame on the same `unique_id`/`ds` grid. For stock level `N`, `cdf(N)` is the service probability `P(Y<=N)` and `sf(N)` is the exceedance risk `P(Y>N)`.

In [4]:
display(count_forecast.ppf([0.50, 0.90, 0.95]).head())
inventory_level = 8
service = count_forecast.cdf(inventory_level)
risk = count_forecast.sf(inventory_level)
probabilities = service.join(risk[[f"P(Y>{inventory_level})"]])
probabilities["probability_check"] = (
    probabilities[f"P(Y<={inventory_level})"] + probabilities[f"P(Y>{inventory_level})"]
)
display(probabilities.head())
count_forecast.pmf([0, 5, 8]).head()

,unique_id,ds,LinearRegression,Q(0.5),Q(0.9),Q(0.95)
0,store_A,2025-05-14,4.420151,3,7,7
1,store_A,2025-05-15,4.512866,2,7,7
2,store_A,2025-05-16,4.912694,2,14,14
3,store_A,2025-05-17,5.357449,5,10,10
4,store_A,2025-05-18,5.902066,8,14,14


,unique_id,ds,LinearRegression,P(Y<=8),P(Y>8),probability_check
0,store_A,2025-05-14,4.420151,1.00000,1.110223e-16,1.0
1,store_A,2025-05-15,4.512866,1.00000,1.110223e-16,1.0
2,store_A,2025-05-16,4.912694,0.80400,1.960002e-01,1.0
3,store_A,2025-05-17,5.357449,0.80400,1.960002e-01,1.0
4,store_A,2025-05-18,5.902066,0.79596,2.040402e-01,1.0


,unique_id,ds,LinearRegression,P(Y=0),P(Y=5),P(Y=8)
0,store_A,2025-05-14,4.420151,0.19998,0.00000,0.00000
1,store_A,2025-05-15,4.512866,0.19600,0.00000,0.00000
2,store_A,2025-05-16,4.912694,0.00000,0.00000,0.00000
3,store_A,2025-05-17,5.357449,0.00000,0.40402,0.19798
4,store_A,2025-05-18,5.902066,0.20200,0.00000,0.39598


### Held-out evaluation and first-stage diagnostics

Both evaluations use the final seven days reserved as a test set. The predictive forecast evaluates complete central intervals, while `FirstStageEvaluator` summarizes point forecasts independently of conformal scaling.

In [5]:
count_test = count_test.sort_values(["unique_id", "ds"]).reset_index(drop=True)
count_observed = count_test["y"].to_numpy()
display(count_forecast.evaluate(count_observed, coverages=[0.80, 0.90, 0.95]))

count_first_stage_test = count_forecast.to_frame().merge(
    count_test[["unique_id", "ds", "y"]],
    on=["unique_id", "ds"],
    how="inner",
    validate="one_to_one",
)
display(FirstStageEvaluator.evaluate(
    count_first_stage_test, prediction_col="LinearRegression"
) )
FirstStageEvaluator.calibration_table(
    count_first_stage_test, prediction_col="LinearRegression", n_bins=5
)

,coverage,coverage_rate,interval_width_mean,mwis
0,0.80,0.786,8.071,11.643
1,0.90,0.786,8.071,15.214
2,0.95,0.786,8.071,22.357


,wape,pbias,score,forecast_instability,false_demand_on_zero_days_avg_pred,peak_demand_deviation
0,0.3245,-0.1473,0.4718,0.1263,0.0,-0.1473


,calibration_bin,count,mean_prediction,mean_observed,mean_residual
0,"(4.419, 5.107]",3,4.615237,5.0,0.384763
1,"(5.107, 5.398]",3,5.320026,7.0,1.679974
2,"(5.398, 5.839]",2,5.557185,6.5,0.942815
3,"(5.839, 6.369]",3,6.124306,5.0,-1.124306
4,"(6.369, 7.415]",3,6.953435,10.0,3.046565


### Optimize discrete inventory

A time-series CPS forecast can be passed directly to the solver. Its panel and underlying distribution are already aligned.

In [6]:
count_plan = NewsvendorSolver.optimize_distribution(
    count_forecast, underage_cost=9.0, overage_cost=1.0
)
display(count_plan.head())
NewsvendorSolver.marginal_benefit_distribution(
    count_forecast, underage_cost=9.0, overage_cost=1.0, units=range(0, 11, 2)
).head()

,unique_id,ds,LinearRegression,critical_ratio,y_optimal
0,store_A,2025-05-14,4.420151,0.9,7.0
1,store_A,2025-05-15,4.512866,0.9,7.0
2,store_A,2025-05-16,4.912694,0.9,14.0
3,store_A,2025-05-17,5.357449,0.9,10.0
4,store_A,2025-05-18,5.902066,0.9,14.0


,unique_id,ds,LinearRegression,MB(k=0),MB(k=2),MB(k=4),MB(k=6),MB(k=8),MB(k=10)
0,store_A,2025-05-14,4.420151,9.0,7.000202,3.020202,0.979800,-1.000000,-1.000000
1,store_A,2025-05-15,4.512866,9.0,4.999596,0.999798,0.999798,-1.000000,-1.000000
2,store_A,2025-05-16,4.912694,9.0,6.959598,2.959800,2.959800,0.960002,0.960002
3,store_A,2025-05-17,5.357449,9.0,6.980002,6.980002,2.939802,2.939802,0.960002
4,store_A,2025-05-18,5.902066,9.0,6.980002,6.980002,6.980002,5.000202,1.040402


## 2. Continuous measurements

Continuous CPS uses the same panel workflow, but retains real-valued support and therefore does not expose a PMF.

In [7]:
def make_continuous_panel(n_periods=140, seed=7):
    rng = np.random.default_rng(seed)
    dates = pd.date_range("2025-01-01", periods=n_periods, freq="D")
    frames = []
    for offset, unique_id in enumerate(["region_A", "region_B"]):
        t = np.arange(n_periods)
        y = (10 + 2 * offset + 0.02 * t + 1.5 * np.sin(2 * np.pi * t / 7)
             + rng.normal(0, 0.8 + 0.2 * offset, n_periods))
        frames.append(pd.DataFrame({"unique_id": unique_id, "ds": dates, "y": y}))
    return pd.concat(frames, ignore_index=True)

continuous_data = make_continuous_panel()
continuous_train = continuous_data.groupby("unique_id", group_keys=False).head(-horizon)
continuous_test = continuous_data.groupby("unique_id", group_keys=False).tail(horizon)

In [8]:
continuous_learner = MLForecast(
    models={"LinearRegression": LinearRegression()},
    freq="D", lags=[1, 7, 14], date_features=["dayofweek"],
)
continuous_cps = ContinuousTimeSeriesConformalPredictiveSystem(
    learner=continuous_learner,
    dispersion_learner=RandomForestRegressor(
        n_estimators=100, min_samples_leaf=3, random_state=43, n_jobs=-1
    ),
).fit(continuous_train, horizon=horizon, n_windows=10, static_features=[], n_jobs=1, step_size=horizon - 3)
continuous_forecast = continuous_cps.predict_distribution(h=horizon)

### Continuous PPF, CDF, SF, intervals, and decisions

The distribution interface is the same as in the discrete case, except that quantiles and optimal quantities remain continuous.

In [9]:
display(continuous_forecast.ppf([0.05, 0.50, 0.95]).head())
display(continuous_forecast.cdf([10, 15]).head())
display(continuous_forecast.sf([10, 15]).head())
display(continuous_forecast.interval(coverage=0.90).head())

continuous_observed = continuous_test.sort_values(["unique_id", "ds"])["y"].to_numpy()
display(continuous_forecast.evaluate(
    continuous_observed, coverages=[0.80, 0.90, 0.95]
) )
NewsvendorSolver.optimize_distribution(
    continuous_forecast, underage_cost=6.0, overage_cost=2.0
).head()

,unique_id,ds,LinearRegression,Q(0.05),Q(0.5),Q(0.95)
0,region_A,2025-05-14,11.759313,10.051685,11.846173,13.265356
1,region_A,2025-05-15,13.744833,11.644143,14.290375,15.749486
2,region_A,2025-05-16,13.177199,11.147516,12.766745,13.676344
3,region_A,2025-05-17,13.499878,11.448926,13.259928,14.412403
4,region_A,2025-05-18,11.521766,9.797436,11.304749,13.095016


,unique_id,ds,LinearRegression,P(Y<=10),P(Y<=15)
0,region_A,2025-05-14,11.759313,0.00000,1.000000
1,region_A,2025-05-15,13.744833,0.00000,0.901537
2,region_A,2025-05-16,13.177199,0.00000,1.000000
3,region_A,2025-05-17,13.499878,0.00000,1.000000
4,region_A,2025-05-18,11.521766,0.20004,1.000000


,unique_id,ds,LinearRegression,P(Y>10),P(Y>15)
0,region_A,2025-05-14,11.759313,1.00000,2.220446e-16
1,region_A,2025-05-15,13.744833,1.00000,9.846273e-02
2,region_A,2025-05-16,13.177199,1.00000,2.220446e-16
3,region_A,2025-05-17,13.499878,1.00000,2.220446e-16
4,region_A,2025-05-18,11.521766,0.79996,2.220446e-16


,unique_id,ds,LinearRegression,Q(0.05),Q(0.95)
0,region_A,2025-05-14,11.759313,10.051685,13.265356
1,region_A,2025-05-15,13.744833,11.644143,15.749486
2,region_A,2025-05-16,13.177199,11.147516,13.676344
3,region_A,2025-05-17,13.499878,11.448926,14.412403
4,region_A,2025-05-18,11.521766,9.797436,13.095016


,coverage,coverage_rate,interval_width_mean,mwis
0,0.80,0.714,2.911,6.250
1,0.90,0.929,4.095,6.437
2,0.95,0.929,4.095,8.779


,unique_id,ds,LinearRegression,critical_ratio,y_optimal
0,region_A,2025-05-14,11.759313,0.75,12.044465
1,region_A,2025-05-15,13.744833,0.75,14.656362
2,region_A,2025-05-16,13.177199,0.75,13.299093
3,region_A,2025-05-17,13.499878,0.75,14.224770
4,region_A,2025-05-18,11.521766,0.75,11.838578


## Support comparison

| Target | CDF | SF | PPF | PMF | Newsvendor output |
|---|---:|---:|---:|---:|---|
| Non-negative integer counts | Yes | Yes | Yes | Yes | Integer |
| Continuous values | Yes | Yes | Yes | No | Continuous |

Unlike the tabular cross-conformal CPS, time-series calibration is performed separately by series and forecast horizon using sequential rolling-origin windows.